In [ ]:
!git clone https://huggingface.co/SprintML/tml26_task2

Cloning into 'tml26_task2'...
remote: Enumerating objects: 392, done.
remote: Counting objects: 100% (389/389), done.
remote: Compressing objects: 100% (387/387), done.
remote: Total 392 (delta 16), reused 0 (delta 0), pack-reused 3 (from 1)
Receiving objects: 100% (392/392), 170.91 KiB | 1.07 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Filtering content: 100% (361/361), 15.10 GiB | 59.99 MiB/s, done.


In [ ]:
!ls tml26_task2

submission.py  suspect_models  target_model  task_template.py


In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
%cd tml26_task2

/content/tml26_task2


In [ ]:
!pip install torch torchvision pandas numpy scikit-learn huggingface_hub safetensors tqdm

In [ ]:
!python task_template.py

Traceback (most recent call last):
  File "/content/tml26_task2/task_template.py", line 24, in <module>
    state_dict = load_file(checkpoint_path, device="cpu")
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/safetensors/torch.py", line 336, in load_file
    with safe_open(filename, framework="pt", device=device) as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: No such file or directory: path/to/your/model_checkpoint.safetensors


In [ ]:
!find . -name "*.safetensors"

./target_model/weights.safetensors
./suspect_models/suspect_197.safetensors
./suspect_models/suspect_334.safetensors
./suspect_models/suspect_279.safetensors
./suspect_models/suspect_234.safetensors
./suspect_models/suspect_219.safetensors
./suspect_models/suspect_223.safetensors
./suspect_models/suspect_185.safetensors
./suspect_models/suspect_244.safetensors
./suspect_models/suspect_260.safetensors
./suspect_models/suspect_027.safetensors
./suspect_models/suspect_016.safetensors
./suspect_models/suspect_073.safetensors
./suspect_models/suspect_001.safetensors
./suspect_models/suspect_061.safetensors
./suspect_models/suspect_126.safetensors
./suspect_models/suspect_300.safetensors
./suspect_models/suspect_172.safetensors
./suspect_models/suspect_071.safetensors
./suspect_models/suspect_096.safetensors
./suspect_models/suspect_249.safetensors
./suspect_models/suspect_092.safetensors
./suspect_models/suspect_011.safetensors
./suspect_models/suspect_006.safetensors
./suspect_models/suspe

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18
from safetensors.torch import load_file

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"In use currently: {device}")

In use currently: cuda


In [ ]:
def make_model():
    model = resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()

    model.fc = nn.Linear(model.fc.in_features, 100)


    return model

print("make_model done")

make_model done


In [ ]:
TARGET_PATH = '/content/tml26_task2/target_model/weights.safetensors'


target_model = make_model()


target_model.load_state_dict(load_file(TARGET_PATH, device='cpu'))
target_model = target_model.to(device)
target_model.eval()


print("Loaded Target models")
print("Number of parameters-", sum(p.numel() for p in target_model.parameters()))

Loaded Target models
Number of parameters- 11220132


In [ ]:
def get_weight_vector(model):
    all_weights = []

    for name, param in model.named_parameters():

        all_weights.append(param.detach().cpu().flatten())

    return torch.cat(all_weights)


target_vec = get_weight_vector(target_model)

print(f"Target weight vector extraction done")
print(f"Vector length is- {len(target_vec):,}")


Target weight vector extraction done
Vector length is- 11,220,132


In [ ]:
def compute_similarity(target_vec, suspect_vec):



    cos_sim = torch.nn.functional.cosine_similarity(
        target_vec.unsqueeze(0),
        suspect_vec.unsqueeze(0),
    ).item()


    l2_dist = torch.norm(target_vec - suspect_vec).item()


    l2_score = 1.0 / (1.0 + l2_dist / len(target_vec))

    combined = 0.7 * cos_sim + 0.3 * l2_score


    return combined, cos_sim, l2_score

print("Similarity function done")


test_score, test_cos, test_l2 = compute_similarity(target_vec, target_vec)
print(f"Target and original- combined={test_score:.4f}, cosine={test_cos:.4f}, l2_score={test_l2:.4f}")

Similarity function done
Target and original- combined=1.0006, cosine=1.0009, l2_score=1.0000


In [ ]:
from torchvision.datasets import CIFAR100
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from tqdm import tqdm

cifar100_mean = (0.5071, 0.4867, 0.4408)
cifar100_std = (0.2675, 0.2565, 0.2761)

probe_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar100_mean, cifar100_std),
])

probe_dataset_full = CIFAR100(
    root="/content/data",
    train=False,
    download=True,
    transform=probe_transform,
)

probe_indices = list(range(10000))
probe_dataset = Subset(probe_dataset_full, probe_indices)

probe_loader = DataLoader(
    probe_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
)

print("Probe dataset-", len(probe_dataset))

100%|██████████| 169M/169M [00:19<00:00, 8.60MB/s]


Probe dataset- 10000


In [ ]:
def collect_logits(model, loader, device):
    model.eval()
    all_logits = []

    with torch.no_grad():
        for images, _ in tqdm(loader):
            images = images.to(device)
            logits = model(images)
            all_logits.append(logits.detach().cpu())

    return torch.cat(all_logits, dim=0)


def calc_behavior_scores(target_logits, suspect_logits):
    target_probs = F.softmax(target_logits, dim=1)
    suspect_probs = F.softmax(suspect_logits, dim=1)

    target_preds = target_logits.argmax(dim=1)
    suspect_preds = suspect_logits.argmax(dim=1)

    agreement_score = (target_preds == suspect_preds).float().mean().item()

    logit_cosine_score = F.cosine_similarity(
        target_logits,
        suspect_logits,
        dim=1,
    ).mean().item()

    prob_cosine_score = F.cosine_similarity(
        target_probs,
        suspect_probs,
        dim=1,
    ).mean().item()

    kl_value = F.kl_div(
        suspect_probs.clamp(min=1e-8).log(),
        target_probs,
        reduction="batchmean",
    ).item()

    kl_score = 1.0 / (1.0 + kl_value)

    return {
        "agreement": agreement_score,
        "logit_cosine": logit_cosine_score,
        "prob_cosine": prob_cosine_score,
        "kl_score": kl_score,
    }


def get_batchnorm_vector(model):
    batchnorm_values = []

    for module in model.modules():
        if isinstance(module, torch.nn.BatchNorm2d):
            batchnorm_values.append(module.running_mean.detach().flatten().cpu())
            batchnorm_values.append(module.running_var.detach().flatten().cpu())

    if len(batchnorm_values) == 0:
        return None

    return torch.cat(batchnorm_values)


def vector_cosine_similarity(first_vector, second_vector):
    if first_vector is None or second_vector is None:
        return 0.0

    first_vector = first_vector.float()
    second_vector = second_vector.float()

    return F.cosine_similarity(
        first_vector.unsqueeze(0),
        second_vector.unsqueeze(0),
    ).item()


print("Behavior and Batchnorm functions are ready.")

Behavior and Batchnorm functions are ready.


In [ ]:
print("Trget logits Collection")

target_logits = collect_logits(target_model, probe_loader, device)
target_bn_vector = get_batchnorm_vector(target_model)

print("Target logits shape-", target_logits.shape)
print("Target batchnorm vector length-", len(target_bn_vector))

Trget logits Collection


100%|██████████| 79/79 [00:04<00:00, 18.36it/s]

Target logits shape- torch.Size([10000, 100])
Target batchnorm vector length- 9600


In [ ]:
feature_cache = {}

def save_activation(name):
    def hook(model, input_tensor, output_tensor):
        feature_cache[name] = output_tensor.detach()
    return hook


def collect_layer_features(model, loader, device, layer_name):
    all_features = []

    model.eval()

    with torch.no_grad():
        for images, _ in tqdm(loader):
            images = images.to(device)
            _ = model(images)

            features = feature_cache[layer_name]
            features = torch.flatten(features, start_dim=1)

            all_features.append(features.cpu())

    return torch.cat(all_features, dim=0)


print("Feature extraction functions are ready.")

Feature extraction functions are ready.


In [ ]:
target_model.layer4.register_forward_hook(
    save_activation("target_layer4")
)

print("Target layer4 hook registered.")

Target layer4 hook registered.


In [ ]:
print("Collection of target layer4 features")

target_layer4_features = collect_layer_features(
    target_model,
    probe_loader,
    device,
    "target_layer4"
)

print("Target layer4 features shape-", target_layer4_features.shape)

Collection of target layer4 features


100%|██████████| 79/79 [00:03<00:00, 25.88it/s]


Target layer4 features shape- torch.Size([10000, 8192])


In [ ]:
import os
import gc

SUSPECTS_FOLDER = "/content/tml26_task2/suspect_models"

results = []
debug_rows = []

print("improved scoring for all 360 models starting, Now, using weights + BatchNorm + behavior")

for i in range(360):
    filename = f"suspect_{i:03d}.safetensors"
    filepath = os.path.join(SUSPECTS_FOLDER, filename)

    if not os.path.exists(filepath):
        results.append({"id": i, "score": 0.0})
        continue

    try:
        suspect_model = make_model()
        suspect_model.load_state_dict(load_file(filepath, device="cpu"), strict=True)
        suspect_model = suspect_model.to(device)
        suspect_model.eval()

        suspect_model.layer4.register_forward_hook(
            save_activation("suspect_layer4")
        )

        # 1. Weight similarity
        suspect_vec = get_weight_vector(suspect_model)
        weight_score, weight_cos, weight_l2 = compute_similarity(target_vec, suspect_vec)

        # 2. BatchNorm similarity
        suspect_bn_vector = get_batchnorm_vector(suspect_model)
        bn_score = vector_cosine_similarity(target_bn_vector, suspect_bn_vector)

        # 3. Behavioral similarity
        suspect_logits = collect_logits(suspect_model, probe_loader, device)
        behavior_scores = calc_behavior_scores(target_logits, suspect_logits)

        suspect_layer4_features = collect_layer_features(
            suspect_model,
            probe_loader,
            device,
            "suspect_layer4"
        )

        feature_score = F.cosine_similarity(
            target_layer4_features,
            suspect_layer4_features,
            dim=1,
        ).mean().item()

        agreement_score = behavior_scores["agreement"]
        logit_cosine_score = behavior_scores["logit_cosine"]
        prob_cosine_score = behavior_scores["prob_cosine"]
        kl_score = behavior_scores["kl_score"]


        final_score = (
    0.10 * weight_score +
    0.15 * bn_score +
    0.20 * logit_cosine_score +
    0.15 * prob_cosine_score +
    0.15 * kl_score +
    0.10 * agreement_score +
    0.15 * feature_score
)

        results.append({
            "id": i,
            "score": final_score,
        })

        debug_rows.append({
            "id": i,
            "final_score": final_score,
            "weight_score": weight_score,
            "bn_score": bn_score,
            "agreement": agreement_score,
            "logit_cosine": logit_cosine_score,
            "prob_cosine": prob_cosine_score,
            "kl_score": kl_score,
            "feature_score": feature_score,
        })

        if i % 10 == 0:
            print(
                f"Processed {i:03d} | "
                f"final={final_score:.4f} | "
                f"w={weight_score:.4f} | "
                f"bn={bn_score:.4f} | "
                f"feat={feature_score:.4f} | "
                f"logit={logit_cosine_score:.4f} | "
                f"prob={prob_cosine_score:.4f} | "
                f"agree={agreement_score:.4f}"
            )

        del suspect_model, suspect_vec, suspect_logits
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as error:
        print(f"Error with suspect {i:03d}: {error}")
        results.append({"id": i, "score": 0.0})

print("Finished improved scoring")

improved scoring for all 360 models starting, Now, using weights + BatchNorm + behavior


100%|██████████| 79/79 [00:06<00:00, 12.48it/s]


Processed 000 | final=0.4620 | w=0.4399 | bn=0.2784 | feat=0.2589 | logit=0.6167 | prob=0.6438 | agree=0.5305


100%|██████████| 79/79 [00:03<00:00, 24.56it/s]


Processed 010 | final=0.1896 | w=0.4346 | bn=0.0807 | feat=0.2337 | logit=0.1787 | prob=0.1695 | agree=0.0852


...

Processed 340 | final=0.6094 | w=0.4684 | bn=0.4168 | feat=0.2439 | logit=0.8711 | prob=0.8171 | agree=0.7095


100%|██████████| 79/79 [00:04<00:00, 19.70it/s]


Processed 350 | final=0.6161 | w=0.4661 | bn=0.4393 | feat=0.2314 | logit=0.8773 | prob=0.8262 | agree=0.7157


100%|██████████| 79/79 [00:03<00:00, 24.70it/s]


Finished improved scoring


In [ ]:
import pandas as pd

df = pd.DataFrame(results)
debug_df = pd.DataFrame(debug_rows)

print("score stats:")
print(df["score"].describe())

print("\nTop 30 most suspicious models:")
print(df.sort_values("score", ascending=False).head(30))

print("\nDebug view for top 30:")
print(debug_df.sort_values("final_score", ascending=False).head(30))

score stats:
count    360.000000
mean       0.623184
std        0.207851
min        0.135088
25%        0.472907
50%        0.622216
75%        0.751285
max        1.000060
Name: score, dtype: float64

Top 30 most suspicious models:
      id     score
358  358  1.000060
124  124  1.000060
71    71  1.000060
145  145  1.000060
138  138  1.000060
224  224  0.996764
259  259  0.994900
78    78  0.993793
313  313  0.993792
168  168  0.993760
14    14  0.993530
341  341  0.993319
148  148  0.993158
45    45  0.992884
104  104  0.992844
195  195  0.992743
218  218  0.991552
54    54  0.990709
198  198  0.989779
8      8  0.989761
300  300  0.989367
110  110  0.988969
236  236  0.988624
136  136  0.987186
35    35  0.985910
318  318  0.983955
174  174  0.979321
221  221  0.978204
348  348  0.974123
276  276  0.971911

Debug view for top 30:
      id  final_score  weight_score  bn_score  agreement  logit_cosine  \
358  358     1.000060      1.000600  1.000001     1.0000      1.000000   
124  1

In [ ]:
rank_columns = [
    "weight_score",
    "bn_score",
    "logit_cosine",
    "prob_cosine",
    "kl_score",
    "agreement",
    "feature_score",
]

for column_name in rank_columns:
    debug_df[f"{column_name}_rank"] = debug_df[column_name].rank(
        ascending=False,
        method="average"
    )

rank_weights = {
    "weight_score_rank": 0.10,
    "bn_score_rank": 0.15,
    "logit_cosine_rank": 0.15,
    "prob_cosine_rank": 0.10,
    "kl_score_rank": 0.10,
    "agreement_rank": 0.10,
    "feature_score_rank": 0.30,
}

debug_df["rank_fusion_score"] = 0.0

for rank_column, weight in rank_weights.items():
    debug_df["rank_fusion_score"] += weight * debug_df[rank_column]

debug_df["rank_fusion_score"] = (
    debug_df["rank_fusion_score"].max() - debug_df["rank_fusion_score"]
)

debug_df["rank_fusion_score"] = (
    debug_df["rank_fusion_score"] - debug_df["rank_fusion_score"].min()
) / (
    debug_df["rank_fusion_score"].max()
    - debug_df["rank_fusion_score"].min()
    + 1e-8
)

print("Rank fusion complete.")
print(debug_df.sort_values("rank_fusion_score", ascending=False).head(30))

Rank fusion complete.
      id  final_score  weight_score  bn_score  agreement  logit_cosine  \
358  358     1.000060      1.000600  1.000001     1.0000      1.000000   
124  124     1.000060      1.000600  1.000001     1.0000      1.000000   
71    71     1.000060      1.000600  1.000001     1.0000      1.000000   
145  145     1.000060      1.000600  1.000001     1.0000      1.000000   
138  138     1.000060      1.000600  1.000001     1.0000      1.000000   
224  224     0.996764      1.000554  1.000001     0.9797      0.996997   
259  259     0.994900      1.000481  1.000001     0.9719      0.994301   
14    14     0.993530      1.000430  1.000001     0.9680      0.992439   
148  148     0.993158      1.000293  1.000001     0.9689      0.990732   
168  168     0.993760      1.000846  0.999904     0.9651      0.997633   
313  313     0.993792      1.000560  0.999932     0.9656      0.997692   
78    78     0.993793      1.000497  0.999933     0.9657      0.997723   
8      8     0.9

In [ ]:
submission_df = debug_df[["id", "rank_fusion_score"]].copy()
submission_df.columns = ["id", "score"]

submission_df.to_csv("submission.csv", index=False)

print("Rank-fusion submission saved.")

check = pd.read_csv("submission.csv")
print(check.head())
print("Total rows:", len(check))
print("Min id:", check["id"].min())
print("Max id:", check["id"].max())
print("Any NaN scores:", check["score"].isna().any())

Rank-fusion submission saved.
   id     score
0   0  0.314474
1   1  0.142251
2   2  0.428509
3   3  0.314327
4   4  0.683845
Total rows: 360
Min id: 0
Max id: 359
Any NaN scores: False


In [ ]:
import requests
import sys

API_KEY = "613bfba4b9ba5a4c6bce1eeecb39f227"
FILE_PATH = "submission.csv"
BASE_URL = "http://34.63.153.158"
TASK_ID = "19-stolen-model-detection"
SUBMIT = True

if SUBMIT:
    if not os.path.isfile(FILE_PATH):
        print("File not found:", FILE_PATH)
    else:
        print("Submitting...")
        with open(FILE_PATH, "rb") as f:
            files = {"file": (os.path.basename(FILE_PATH), f, "csv")}
            resp = requests.post(
                f"{BASE_URL}/submit/{TASK_ID}",
                headers={"X-API-Key": API_KEY},
                files=files,
                timeout=(10, 120),
            )

        try:
            body = resp.json()
        except:
            body = {"raw_text": resp.text}

        print("Server response:", body)

Submitting...
Server response: {'submission_id': 2022, 'status': 'success', 'message': 'Submission evaluated successfully! Check the leaderboard to see your score.'}
